# Book 1: Solutions to every "Try it yourself" exercise
**Companion notebook** · Book 1: Python and Data Foundations for ML

Run the set-up cell first, then any chapter's solutions in order. (Chapter 6 has its own solutions inside the chapter.)

## Set-up: load the data once for all chapters

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

bank_data = fetch_ucirepo(id=222)
bank = bank_data.data.features.copy()
bank["subscribed"] = bank_data.data.targets["y"]
bank = bank.fillna("unknown")
print(bank.shape)

## Chapter 2

In [ ]:
print(bank.columns.tolist())

In [ ]:
print(bank["job"].value_counts().head(3))
print(bank["marital"].value_counts())

In [ ]:
print("Mean balance:  ", round(bank["balance"].mean(), 2))
print("Median balance:", bank["balance"].median())

## Chapter 3

In [ ]:
name = "Priya"
age = 33
balance = 1506.50
has_housing_loan = True

for value in [name, age, balance, has_housing_loan]:
    print(value, type(value))

In [ ]:
def fd_maturity(amount, rate, years):
    return amount * (1 + rate) ** years

print(round(fd_maturity(10000, 0.07, 5), 2))

In [ ]:
jobs = [" Management", "TECHNICIAN ", "blue-Collar", "  admin."]

clean_jobs = []
for job in jobs:
    clean_jobs.append(job.strip().lower())

print(clean_jobs)

In [ ]:
def balance_label(balance):
    if balance < 0:
        return "Overdrawn"
    elif balance < 1000:
        return "Low"
    elif balance < 10000:
        return "Medium"
    else:
        return "High"

for b in [-8019, 448, 1506, 102127]:
    print(b, balance_label(b))

In [ ]:
all_customers = bank.to_dict("records")

marital_counts = {}
for customer in all_customers:
    status = customer["marital"]
    marital_counts[status] = marital_counts.get(status, 0) + 1

print(marital_counts)

In [ ]:
def check_loan_eligibility(customer):
    if customer["age"] < 21 or customer["age"] > 60:
        return False, "Age outside 21 to 60"
    if customer["default"] == "yes":
        return False, "Has defaulted on credit before"
    if customer["loan"] == "yes":
        return False, "Already has a personal loan"
    if customer["balance"] < 500:
        return False, "Balance below 500"
    return True, "Eligible"


def check_loan_eligibility_v2(customer):
    if customer["job"] == "unknown":
        return False, "Job unknown"
    return check_loan_eligibility(customer)


before = sum(1 for c in all_customers if check_loan_eligibility(c)[0])
after = sum(1 for c in all_customers if check_loan_eligibility_v2(c)[0])
print(f"Eligible before: {before:,}")
print(f"Eligible after:  {after:,}")
print(f"Change: {before - after}")

In [ ]:
top_five = sorted(all_customers, key=lambda c: c["balance"], reverse=True)[:5]
for c in top_five:
    print(c["age"], c["job"], c["balance"])

## Chapter 4

In [ ]:
# Setup: recreate the Chapter 4 mini project file and classes
import csv
import json
from datetime import date

bank.to_csv("bank_customers.csv", index=False)
with open("bank_customers.csv", "a") as file:
    file.write("thirty,technician,married,secondary,no,1200,yes,no,cellular,5,may,100,1,-1,0,unknown,no\n")
    file.write("41,admin.,single,tertiary,no,,no,no,cellular,5,may,100,1,-1,0,unknown,yes\n")


class BankCustomer:
    """One bank customer from the Bank Marketing dataset."""

    def __init__(self, age, job, balance, has_loan, subscribed):
        if age < 18 or age > 120:
            raise ValueError(f"Unrealistic age: {age}")
        self.age = age
        self.job = job
        self.balance = balance
        self.has_loan = has_loan
        self.subscribed = subscribed

    def is_overdrawn(self):
        return self.balance < 0

    def __str__(self):
        return f"BankCustomer(age={self.age}, job={self.job}, balance={self.balance})"


def customer_from_row(row):
    return BankCustomer(
        age=int(row["age"]),
        job=row["job"],
        balance=int(row["balance"]),
        has_loan=row["loan"] == "yes",
        subscribed=row["subscribed"] == "yes",
    )


def load_customers(path):
    customers = []
    skipped = 0
    with open(path, newline="") as file:
        reader = csv.DictReader(file)
        for line_number, row in enumerate(reader, start=2):
            try:
                customers.append(customer_from_row(row))
            except ValueError as error:
                skipped += 1
                print(f"Skipped line {line_number}: {error}")
    return customers, skipped


customers, skipped = load_customers("bank_customers.csv")
print(len(customers), skipped)

In [ ]:
loan_ages = [c.age for c in customers if c.has_loan]
print(len(loan_ages), loan_ages[:5])

first_jobs = [c.job for c in customers[:20]]
job_counts = {job: first_jobs.count(job) for job in set(first_jobs)}
print(dict(sorted(job_counts.items())))

In [ ]:
def safe_ratio(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print(f"Cannot divide {a} by zero")
        return None

print(safe_ratio(10, 4))
print(safe_ratio(10, 0))

In [ ]:
def log_call(customer_name, outcome):
    line = f"{date.today().isoformat()} | {customer_name} | {outcome}\n"
    with open("call_log.txt", "a") as file:
        file.write(line)

log_call("Priya", "subscribed")
log_call("Arjun", "not interested")
log_call("Meera", "call back")

with open("call_log.txt") as file:
    print(file.read())

In [ ]:
class BankCustomer(BankCustomer):
    OVERDRAFT_LIMIT = -1000

    def withdraw(self, amount):
        if self.balance - amount < self.OVERDRAFT_LIMIT:
            raise ValueError(
                f"Withdrawal refused: balance would be {self.balance - amount}, "
                f"below the limit of {self.OVERDRAFT_LIMIT}"
            )
        self.balance -= amount
        return self.balance

    def monthly_fee(self):
        return 5


priya = BankCustomer(33, "technician", 200, False, False)
print(priya.withdraw(1000))
try:
    priya.withdraw(500)
except ValueError as error:
    print("Error:", error)

In [ ]:
class StudentCustomer(BankCustomer):
    def __init__(self, age, job, balance, has_loan, subscribed, college):
        super().__init__(age, job, balance, has_loan, subscribed)
        self.college = college

    def monthly_fee(self):
        return 0


sana = StudentCustomer(20, "student", 300, False, False, "City College")
print(sana, sana.college)
print("Student fee:", sana.monthly_fee(), " Standard fee:", priya.monthly_fee())
print(isinstance(sana, BankCustomer))

In [ ]:
%%writefile bank_loader.py
"""Load Bank Marketing customers from a CSV file."""
import csv


class BankCustomer:
    def __init__(self, age, job, balance, has_loan, subscribed):
        if age < 18 or age > 120:
            raise ValueError(f"Unrealistic age: {age}")
        self.age = age
        self.job = job
        self.balance = balance
        self.has_loan = has_loan
        self.subscribed = subscribed


def customer_from_row(row):
    return BankCustomer(
        age=int(row["age"]),
        job=row["job"],
        balance=int(row["balance"]),
        has_loan=row["loan"] == "yes",
        subscribed=row["subscribed"] == "yes",
    )


def load_customers(path):
    customers = []
    skipped = 0
    with open(path, newline="") as file:
        for row in csv.DictReader(file):
            try:
                customers.append(customer_from_row(row))
            except ValueError:
                skipped += 1
    return customers, skipped

In [ ]:
from bank_loader import load_customers as load_from_module

loaded, skipped_rows = load_from_module("bank_customers.csv")
print(len(loaded), skipped_rows)

In [ ]:
import logging

logging.basicConfig(filename="pipeline.log", level=logging.WARNING,
                    format="%(levelname)s: %(message)s", force=True)

customers = []
skipped = 0
with open("bank_customers.csv", newline="") as file:
    for line_number, row in enumerate(csv.DictReader(file), start=2):
        try:
            customers.append(customer_from_row(row))
        except ValueError as error:
            skipped += 1
            logging.warning(f"Skipped line {line_number}: {error}")


def rate(group):
    return round(sum(c.subscribed for c in group) / len(group), 3)


with_loan = [c for c in customers if c.has_loan]
without_loan = [c for c in customers if not c.has_loan]

report = {
    "customers_loaded": len(customers),
    "rows_skipped": skipped,
    "subscription_rate": rate(customers),
    "subscription_rate_by_loan": {"with_loan": rate(with_loan), "without_loan": rate(without_loan)},
}
print(json.dumps(report, indent=2))

with open("pipeline.log") as file:
    print(file.read())

## Chapter 5

In [ ]:
jobs_yes = set(bank.loc[bank["subscribed"] == "yes", "job"])
jobs_no = set(bank.loc[bank["subscribed"] == "no", "job"])

print(len(jobs_yes), len(jobs_no))
print("Only among subscribers:", jobs_yes - jobs_no)
print("Only among non-subscribers:", jobs_no - jobs_yes)

In [ ]:
from collections import Counter

education_counts = Counter(bank["education"])
marital_counts = Counter(bank["marital"])

print(education_counts.most_common(3))
print(marital_counts.most_common()[-1])

In [ ]:
def count_items(nested):
    total = 0
    for item in nested:
        if isinstance(item, list):
            total += count_items(item)
        else:
            total += 1
    return total

print(count_items([1, [2, 3], [4, [5, 6]]]))
print(count_items([]))

In [ ]:
import functools

def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper


@count_calls
def greet(name):
    return f"Hello, {name}"

greet("Priya")
greet("Arjun")
greet("Meera")
print(greet.calls)

In [ ]:
def batches(items, size):
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

for batch in batches(range(10), 3):
    print(batch)

In [ ]:
from dataclasses import dataclass

@dataclass
class BankCustomer:
    name: str
    age: int
    job: str
    balance: int

    @property
    def is_overdrawn(self):
        return self.balance < 0


ravi = BankCustomer("Ravi", 41, "services", -250)
print(ravi)
print(ravi.is_overdrawn)

In [ ]:
import os
from contextlib import contextmanager

@contextmanager
def working_directory(path):
    original = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(original)


os.makedirs("reports", exist_ok=True)
start = os.getcwd()
try:
    with working_directory("reports"):
        print("Inside:", os.path.basename(os.getcwd()))
        raise RuntimeError("something went wrong")
except RuntimeError as error:
    print("Caught:", error)
print("Back where we started:", os.getcwd() == start)

In [ ]:
import re
from datetime import datetime

def extract_dates(text):
    found = re.findall(r"\b\d{2}/\d{2}/\d{4}\b", text)
    return [datetime.strptime(d, "%d/%m/%Y").date() for d in found]

note = "Called on 05/05/2026, follow up on 21/06/2026. Ref 123/45/6789 is not a date."
print(extract_dates(note))

Set-up: recreate the Chapter 5 toolkit file (`text_cleaning.py`), exactly as in the Chapter 5 mini project.

In [ ]:
%%writefile text_cleaning.py
"""A small toolkit for cleaning messy customer text data."""
import re
from collections import Counter
from dataclasses import dataclass


def clean_category(text: str) -> str:
    cleaned = re.sub(r"[^a-z]+", "-", text.lower()).strip("-")
    if not cleaned:
        raise ValueError(f"No letters found in {text!r}")
    return cleaned


def parse_amount(text: str) -> float:
    match = re.search(r"-?[\d,]+(\.\d+)?", text)
    if match is None:
        raise ValueError(f"No amount found in {text!r}")
    return float(match.group().replace(",", ""))


def mask_phone(text: str) -> str:
    return re.sub(r"\b\d{5}-?\d{5}\b", "XXXXX-XXXXX", text)


@dataclass
class CleaningReport:
    rows: int
    variants_before: int
    categories_after: int


def clean_column(values: list[str]) -> tuple[list[str], CleaningReport]:
    cleaned = [clean_category(v) for v in values]
    report = CleaningReport(
        rows=len(values),
        variants_before=len(Counter(values)),
        categories_after=len(Counter(cleaned)),
    )
    return cleaned, report

In [ ]:
%%writefile test_text_cleaning_extra.py
import pytest
from text_cleaning import clean_category, parse_amount, mask_phone


def test_clean_category_empty_string_raises():
    with pytest.raises(ValueError):
        clean_category("")


def test_parse_amount_takes_first_number():
    assert parse_amount("paid 200, owes 50") == 200.0


def test_mask_phone_leaves_short_numbers_alone():
    assert mask_phone("PIN 12345") == "PIN 12345"


def test_clean_category_accented_letters():
    assert clean_category("Café Owner") == "cafe-owner"

In [ ]:
!python -m pytest -q test_text_cleaning_extra.py

## Chapter 7

In [ ]:
print(bank.groupby("subscribed")["age"].mean().round(1))

In [ ]:
mask = (bank["balance"].between(1000, 5000)
        & (bank["housing"] == "no")
        & (bank["loan"] == "no"))
print(mask.sum())

In [ ]:
print(bank.groupby("job")["balance"].median().sort_values(ascending=False).head(5))

In [ ]:
print(pd.crosstab(bank["education"], bank["subscribed"], normalize="index").round(3))

In [ ]:
bank["calls_band"] = pd.cut(bank["campaign"], bins=[0, 1, 3, 100], labels=["1", "2-3", "4+"])
bank["subscribed_num"] = (bank["subscribed"] == "yes").astype(int)
print(bank.groupby("calls_band", observed=True)["subscribed_num"].agg(["size", "mean"]).round(3))

In [ ]:
months = pd.DataFrame({
    "month": ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"],
    "month_num": range(1, 13),
})
merged = bank.merge(months, on="month", how="left")
print(merged.groupby(["month_num", "month"]).size())

In [ ]:
combo = bank.groupby(["job", "marital"])["subscribed_num"].agg(["size", "mean"])
combo = combo[combo["size"] >= 500].sort_values("mean", ascending=False)
print(combo.head(3).round(3))

## Chapter 8

In [ ]:
import seaborn as sns

order = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
month_rates = bank.groupby("month")["subscribed_num"].mean().reindex(order) * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(month_rates.index, month_rates.values)
ax.set_title("Subscription rate by month")
ax.set_xlabel("Month")
ax.set_ylabel("Subscription rate (%)")
plt.show()

print("Highest:", month_rates.idxmax(), round(month_rates.max(), 1))
print("Lowest: ", month_rates.idxmin(), round(month_rates.min(), 1))

In [ ]:
short_calls = bank[bank["duration"] <= 1500]
median_duration = bank["duration"].median()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(short_calls["duration"], bins=50, edgecolor="white")
ax.axvline(median_duration, color="black", linestyle="--", label=f"Median: {median_duration:.0f} s")
ax.set_title("Most calls last a few minutes")
ax.set_xlabel("Call duration (seconds), up to 1,500")
ax.set_ylabel("Number of calls")
ax.legend()
plt.show()

print("Calls left out:", len(bank) - len(short_calls))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=bank, x="subscribed", y="balance", showfliers=False, ax=ax)
ax.set_title("Subscribers have higher balances")
ax.set_xlabel("Subscribed")
ax.set_ylabel("Balance (euros)")
plt.show()

print(bank.groupby("subscribed")["balance"].median())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=bank, y="education", hue="subscribed", ax=axes[0])
axes[0].set_title("Counts: dominated by group size")

edu_rates = bank.groupby("education")["subscribed_num"].mean().sort_values() * 100
axes[1].barh(edu_rates.index, edu_rates.values)
axes[1].set_title("Rates: tertiary education subscribes most")
axes[1].set_xlabel("Subscription rate (%)")
fig.tight_layout()
plt.show()

print(edu_rates.round(1))

In [ ]:
contact_rates = bank.groupby("contact")["subscribed_num"].mean().sort_values(ascending=False) * 100
overall = bank["subscribed_num"].mean() * 100
colours = ["#d62728" if c == contact_rates.idxmax() else "#9e9e9e" for c in contact_rates.index]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(contact_rates.index, contact_rates.values, color=colours)
ax.axhline(overall, color="black", linestyle="--", label=f"All customers: {overall:.1f}%")
ax.bar_label(bars, fmt="%.1f%%", label_type="center", color="white", fontweight="bold")
ax.legend(frameon=False)
ax.grid(False)
ax.set_title("Cellular contacts subscribe at over three times the 'unknown' rate")
ax.set_ylabel("Subscription rate (%)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print(contact_rates.round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(month_rates.index, month_rates.values)
axes[0].set_title("Rate (%) by month")

axes[1].hist(short_calls["duration"], bins=50, edgecolor="white")
axes[1].axvline(median_duration, color="black", linestyle="--")
axes[1].set_title("Call duration (s)")

axes[2].bar(contact_rates.index, contact_rates.values, color=colours)
axes[2].axhline(overall, color="black", linestyle="--")
axes[2].set_title("Rate (%) by contact type")

fig.suptitle("Bank Marketing campaign: my dashboard", fontsize=16)
fig.tight_layout()
fig.savefig("my_dashboard.png", dpi=150)
plt.show()

import os
print(os.path.exists("my_dashboard.png"))

## Chapter 9

In [ ]:
d = bank["duration"]
print("Mean:  ", round(d.mean(), 1))
print("Median:", d.median())
print("Std:   ", round(d.std(), 1))
print("IQR:   ", d.quantile(0.75) - d.quantile(0.25))

In [ ]:
print(pd.crosstab(bank["contact"], bank["subscribed"], normalize="index").round(3))

In [ ]:
p_student = (bank["job"] == "student").mean()
p_yes = (bank["subscribed"] == "yes").mean()
p_both = ((bank["job"] == "student") & (bank["subscribed"] == "yes")).mean()

print(f"P(student) = {p_student:.4f}")
print(f"P(subscribe) = {p_yes:.4f}")
print(f"P(student and subscribe) = {p_both:.4f}")
print(f"P(student) x P(subscribe) = {p_student * p_yes:.4f}")

In [ ]:
z = (bank["balance"] - bank["balance"].mean()) / bank["balance"].std()
print("Above 3:", (z > 3).sum())
print("Below -3:", (z < -3).sum())
print("Lowest z:", round(z.min(), 2), " Highest z:", round(z.max(), 2))

In [ ]:
calls = bank[bank["campaign"] <= 10].groupby("campaign")["subscribed_num"].mean() * 100
x = calls.index.to_numpy(dtype=float)
y = calls.to_numpy()


def train(learning_rate, steps=5000):
    m, b = 0.0, 0.0
    losses = []
    for step in range(steps):
        error = m * x + b - y
        m = m - learning_rate * 2 * np.mean(error * x)
        b = b - learning_rate * 2 * np.mean(error)
        losses.append(np.mean((m * x + b - y) ** 2))
    return losses


for lr in [0.001, 0.03]:
    with np.errstate(all="ignore"):
        losses = train(lr)
    print(f"learning rate {lr}: after 10 = {losses[9]:.3g}, after 100 = {losses[99]:.3g}, after 5000 = {losses[-1]:.3g}")

Set-up: the `permutation_test` function from the Chapter 9 mini project.

In [ ]:
def permutation_test(group_a, group_b, n_shuffles=1000, seed=42):
    rng = np.random.default_rng(seed)
    combined = np.concatenate([group_a, group_b])
    n_a = len(group_a)
    observed = group_a.mean() - group_b.mean()
    count = 0
    for i in range(n_shuffles):
        shuffled = rng.permutation(combined)
        diff = shuffled[:n_a].mean() - shuffled[n_a:].mean()
        if abs(diff) >= abs(observed):
            count += 1
    return observed, count / n_shuffles

In [ ]:
married = bank.loc[bank["marital"] == "married", "subscribed_num"].to_numpy()
single = bank.loc[bank["marital"] == "single", "subscribed_num"].to_numpy()
diff, p = permutation_test(single, married)
print(f"Single {single.mean():.1%} vs married {married.mean():.1%}: difference {diff:.1%}, p-value {p}")

## Chapter 10

In [ ]:
raw = bank_data.data.features.copy()
raw["subscribed"] = bank_data.data.targets["y"]
raw = raw.replace("unknown", np.nan)

flagged = raw.copy()
flagged["contact_missing"] = flagged["contact"].isna().astype(int)
flagged["contact"] = flagged["contact"].fillna(flagged["contact"].mode()[0])

print(flagged["contact"].value_counts())
print(flagged.groupby("contact_missing")["subscribed"].apply(lambda s: (s == "yes").mean()).round(3))

In [ ]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

low, high = iqr_bounds(raw["previous"])
print("Q1, Q3:", raw["previous"].quantile(0.25), raw["previous"].quantile(0.75))
print("Normal range:", low, "to", high)
print("Flagged as outliers:", ((raw["previous"] < low) | (raw["previous"] > high)).sum())

In [ ]:
marital = pd.Series([" Married", "single ", "MARRIED", "Divorced", "single"])
marital = marital.str.strip().str.lower()
print(marital.value_counts())

In [ ]:
levels = raw["education"].map({"primary": 1, "secondary": 2, "tertiary": 3})
print("Missing before:", levels.isna().sum())
levels = levels.fillna(levels.median())
print(levels.value_counts().sort_index())

In [ ]:
groups = raw.copy()
groups["balance_group"] = pd.cut(groups["balance"], bins=[-10000, 0, 1000, 5000, 200000],
                                 labels=["overdrawn or zero", "low", "medium", "high"])
print(groups.groupby("balance_group", observed=True)["subscribed"]
      .agg(customers="size", rate=lambda s: round((s == "yes").mean(), 3)))

In [ ]:
# Setup: the clean_bank function from the Chapter 10 mini project
def clean_bank(raw):
    """Turn the raw bank table into a numeric, model-ready table."""
    df = raw.copy()

    # 1. Missing values
    df["contacted_before"] = (df["pdays"] != -1).astype(int)
    df["poutcome"] = np.where(df["contacted_before"] == 0, "not_contacted",
                              df["poutcome"].fillna("unknown"))
    df["contact"] = df["contact"].fillna("unknown")
    df["education"] = df["education"].fillna("unknown")
    df["job"] = df["job"].fillna(df["job"].mode()[0])

    # 2. Outliers and skew
    df["campaign"] = df["campaign"].clip(upper=df["campaign"].quantile(0.99))
    df["balance"] = np.sign(df["balance"]) * np.log1p(df["balance"].abs())

    # 3. New features
    month_order = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
                   "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
    df["month"] = df["month"].map(month_order)

    # 4. Encoding
    for col in ["default", "housing", "loan", "subscribed"]:
        df[col] = df[col].map({"yes": 1, "no": 0})
    df = pd.get_dummies(df, columns=["job", "marital", "education", "contact", "poutcome"], dtype=int)

    # 5. Remove leakage
    df = df.drop(columns=["duration"])
    return df

In [ ]:
def clean_bank_v2(raw):
    """clean_bank plus a has_any_loan feature."""
    df = clean_bank(raw)
    df["has_any_loan"] = ((df["housing"] == 1) | (df["loan"] == 1)).astype(int)
    return df

v2 = clean_bank_v2(raw)
print("Shape:", v2.shape)
print("Missing values:", v2.isna().sum().sum())
print(v2.groupby("has_any_loan")["subscribed"].mean().round(3))

---
**Dataset credit:** S. Moro, P. Rita and P. Cortez, *Bank Marketing* dataset, UCI Machine Learning Repository (2014), https://doi.org/10.24432/C5K306. Licensed under CC BY 4.0.